# Aula 15 — ROC, Precision-Recall, thresholds e calibração

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/15-roc-pr-threshold-calibracao-laboratorio.ipynb)

**Objetivo:** separar três propriedades de um classificador: ranking, probabilidade e decisão.

Hipóteses pré-registradas:

1. uma transformação monotônica preservará ROC-AUC e average precision;
2. a mesma transformação poderá piorar Brier e log-loss;
3. calibração sigmoid, ajustada fora do treino do modelo-base, recuperará parte da qualidade probabilística;
4. um threshold escolhido por custo na validação reduzirá o custo no teste em relação a 0,5, sem consultar os rótulos do teste.


## Ambiente e dependências

- Python ≥ 3.10
- NumPy ≥ 1.24
- Matplotlib ≥ 3.7
- scikit-learn ≥ 1.3

Os dados são sintéticos e gerados com seed fixa. Não há download, credencial ou estado externo.


In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("error")
SEED = 20260908
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 1. ROC-AUC calculada por pares

Para cada par positivo-negativo, contamos 1 se o positivo tiver score maior e 0,5 em empate. O exemplo da aula deve produzir $3/4=0{,}75$.


In [ ]:
y_small = np.array([1, 0, 1, 0])
s_small = np.array([0.9, 0.8, 0.7, 0.1])

def auc_by_pairs(y, score):
    positive = score[y == 1]
    negative = score[y == 0]
    wins = sum((p > n) + 0.5 * (p == n) for p in positive for n in negative)
    return float(wins / (len(positive) * len(negative)))

auc_manual = auc_by_pairs(y_small, s_small)
auc_library = roc_auc_score(y_small, s_small)
print(f"AUC manual: {auc_manual:.6f}")
print(f"AUC scikit-learn: {auc_library:.6f}")
assert auc_manual == 0.75
assert np.isclose(auc_manual, auc_library)

## 2. Dados e protocolo sem leakage

A unidade é um evento independente. Criamos 5.000 eventos com 8% de positivos e separamos:

- 60% para ajustar o modelo-base;
- 20% para calibrar e escolher a política;
- 20% como teste final intocado.

O exemplo simula risco operacional; não pretende reproduzir uma população específica.


In [ ]:
X, y = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    n_repeated=0,
    n_clusters_per_class=2,
    weights=[0.92, 0.08],
    class_sep=1.1,
    flip_y=0.01,
    random_state=SEED,
)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)
X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED + 1
)

for name, target in [("treino", y_train), ("calibração", y_cal), ("teste", y_test)]:
    print(f"{name:11s}: n={len(target):4d}, prevalência={target.mean():.6f}")

assert len(y_train) == 3000 and len(y_cal) == len(y_test) == 1000
assert max(y_train.mean(), y_cal.mean(), y_test.mean()) - min(
    y_train.mean(), y_cal.mean(), y_test.mean()
) < 0.002

## 3. Modelo-base e distorção monotônica

O scaler é ajustado apenas no treino, dentro de um `Pipeline`. Depois, aplicamos aos scores uma função logística monotônica forte. Ela preserva a ordem, mas muda o significado probabilístico.


In [ ]:
base_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=SEED),
)
base_model.fit(X_train, y_train)

p_base_cal = base_model.predict_proba(X_cal)[:, 1]
p_base_test = base_model.predict_proba(X_test)[:, 1]

EPS = 1e-8
def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def expit(z):
    return 1 / (1 + np.exp(-z))

def distort(p):
    return expit(2.6 * logit(p) + 1.2)

p_bad_cal = distort(p_base_cal)
p_bad_test = distort(p_base_test)

assert np.array_equal(np.argsort(p_base_test), np.argsort(p_bad_test))
print("Ordem preservada em todos os casos de teste.")

## 4. Ranking versus qualidade probabilística

ROC-AUC e AP usam a ordem; Brier e log-loss usam os valores probabilísticos. Uma transformação monotônica deve deixar as duas primeiras invariantes e pode alterar as duas últimas.


In [ ]:
def probabilistic_metrics(y_true, p):
    return {
        "roc_auc": roc_auc_score(y_true, p),
        "average_precision": average_precision_score(y_true, p),
        "brier": brier_score_loss(y_true, p),
        "log_loss": log_loss(y_true, p),
    }

metrics_base = probabilistic_metrics(y_test, p_base_test)
metrics_bad = probabilistic_metrics(y_test, p_bad_test)

for name, values in [("base", metrics_base), ("distorcido", metrics_bad)]:
    print(name, {k: round(v, 6) for k, v in values.items()})

assert np.isclose(metrics_base["roc_auc"], metrics_bad["roc_auc"])
assert np.isclose(metrics_base["average_precision"], metrics_bad["average_precision"])
assert metrics_bad["brier"] > metrics_base["brier"]
assert metrics_bad["log_loss"] > metrics_base["log_loss"]

### Curvas ROC e PR

A linha horizontal na PR mostra a prevalência do teste, referência esperada para um ranking aleatório. Ela não é uma meta universal: muda com a população.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_base_test)
precision, recall, _ = precision_recall_curve(y_test, p_base_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(fpr, tpr, label=f"ROC-AUC = {metrics_base['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="ranking aleatório")
axes[0].set(xlabel="FPR", ylabel="TPR / recall", title="Curva ROC")
axes[0].legend()

axes[1].plot(recall, precision, label=f"AP = {metrics_base['average_precision']:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", label=f"prevalência = {y_test.mean():.3f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Curva Precision-Recall", ylim=(0, 1.02))
axes[1].legend()
fig.suptitle("Mesmo ranking para probabilidades-base e distorcidas")
fig.tight_layout()
plt.show()

## 5. Calibração sigmoid em conjunto separado

O calibrador recebe apenas o logit do score distorcido no conjunto de calibração. Trata-se de uma versão explícita do princípio de Platt scaling. O teste permanece intocado.


In [ ]:
calibrator = LogisticRegression(C=1e6, max_iter=2000, random_state=SEED)
calibrator.fit(logit(p_bad_cal).reshape(-1, 1), y_cal)

p_calibrated_cal = calibrator.predict_proba(logit(p_bad_cal).reshape(-1, 1))[:, 1]
p_calibrated_test = calibrator.predict_proba(logit(p_bad_test).reshape(-1, 1))[:, 1]

metrics_calibrated = probabilistic_metrics(y_test, p_calibrated_test)
print("coeficiente:", round(float(calibrator.coef_[0, 0]), 6))
print("intercepto:", round(float(calibrator.intercept_[0]), 6))
print("calibrado", {k: round(v, 6) for k, v in metrics_calibrated.items()})

assert np.isclose(metrics_bad["roc_auc"], metrics_calibrated["roc_auc"])
assert metrics_calibrated["brier"] < metrics_bad["brier"]
assert metrics_calibrated["log_loss"] < metrics_bad["log_loss"]

### Diagrama de confiabilidade

Usamos bins por quantis para distribuir aproximadamente o mesmo número de casos em cada ponto. O histograma abaixo impede interpretar bins pouco povoados como evidência forte.


In [ ]:
def reliability(y_true, p, bins=10):
    observed, predicted = calibration_curve(y_true, p, n_bins=bins, strategy="quantile")
    return predicted, observed

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.4, 1]})
for label, probs in [
    ("base", p_base_test),
    ("distorcido", p_bad_test),
    ("recalibrado", p_calibrated_test),
]:
    predicted, observed = reliability(y_test, probs)
    axes[0].plot(predicted, observed, marker="o", label=label)
axes[0].plot([0, 1], [0, 1], "--", color="black", label="ideal")
axes[0].set(xlabel="Probabilidade média prevista", ylabel="Frequência positiva", title="Confiabilidade")
axes[0].legend()

axes[1].hist(p_calibrated_test, bins=np.linspace(0, 1, 21), color="#4472C4", edgecolor="white")
axes[1].set(xlabel="Probabilidade recalibrada", ylabel="Número de casos", title="Distribuição das previsões")
fig.tight_layout()
plt.show()

## 6. Threshold escolhido por custo na validação

Definimos antes de olhar o teste: falso positivo custa 1 unidade; falso negativo custa 12. Percorremos uma grade fixa de thresholds e escolhemos o menor custo médio na validação. Em empate, preferimos o maior threshold, política mais conservadora em volume.


In [ ]:
C_FP, C_FN = 1.0, 12.0
threshold_grid = np.linspace(0.0, 1.0, 1001)

def decision_counts(y_true, p, threshold):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def mean_cost(y_true, p, threshold):
    counts = decision_counts(y_true, p, threshold)
    return (C_FP * counts["fp"] + C_FN * counts["fn"]) / len(y_true)

validation_costs = np.array([mean_cost(y_cal, p_calibrated_cal, t) for t in threshold_grid])
minimum = validation_costs.min()
best_candidates = threshold_grid[np.isclose(validation_costs, minimum)]
selected_threshold = float(best_candidates.max())
theoretical_threshold = C_FP / (C_FP + C_FN)

print(f"threshold teórico: {theoretical_threshold:.6f}")
print(f"threshold selecionado: {selected_threshold:.6f}")
print(f"custo mínimo na validação: {minimum:.6f}")
assert 0 < selected_threshold < 1

A fórmula teórica supõe probabilidades perfeitas e custos completos. A seleção empírica incorpora erro de estimação e granularidade da amostra. O valor escolhido agora é congelado.


In [ ]:
def policy_report(y_true, p, threshold):
    counts = decision_counts(y_true, p, threshold)
    precision = counts["tp"] / max(counts["tp"] + counts["fp"], 1)
    recall = counts["tp"] / max(counts["tp"] + counts["fn"], 1)
    return {
        "threshold": threshold,
        **counts,
        "precision": precision,
        "recall": recall,
        "mean_cost": mean_cost(y_true, p, threshold),
        "alert_rate": (counts["tp"] + counts["fp"]) / len(y_true),
    }

report_selected = policy_report(y_test, p_calibrated_test, selected_threshold)
report_default = policy_report(y_test, p_calibrated_test, 0.5)

for name, report in [("selecionado", report_selected), ("threshold 0,5", report_default)]:
    printable = {k: round(v, 6) if isinstance(v, float) else v for k, v in report.items()}
    print(name, printable)

assert report_selected["mean_cost"] < report_default["mean_cost"]

## 7. Capacidade fixa: threshold como quantil

Se a equipe pode revisar apenas 5% dos casos, a política é selecionar os maiores scores. O threshold é determinado pela capacidade, e empates na fronteira precisam de regra explícita.


In [ ]:
capacity_fraction = 0.05
k = int(np.ceil(capacity_fraction * len(y_test)))
order = np.argsort(-p_calibrated_test, kind="stable")
selected = order[:k]
tp_at_k = int(y_test[selected].sum())
precision_at_k = tp_at_k / k
recall_at_k = tp_at_k / y_test.sum()

print(f"capacidade: {k} de {len(y_test)} casos")
print(f"precision@{k}: {precision_at_k:.6f}")
print(f"recall@{k}: {recall_at_k:.6f}")
assert len(np.unique(selected)) == k

## 8. Verificações finais e conclusão

Os asserts seguintes funcionam como testes do protocolo e das hipóteses. Eles não provam validade externa, mas detectam regressões mecânicas no laboratório.


In [ ]:
assert np.isfinite(p_calibrated_test).all()
assert ((0 <= p_calibrated_test) & (p_calibrated_test <= 1)).all()
assert len(y_train) + len(y_cal) + len(y_test) == len(y)
assert set(map(tuple, X_train)).isdisjoint(set(map(tuple, X_test)))
assert metrics_base["average_precision"] > y_test.mean()
assert abs(metrics_base["roc_auc"] - metrics_bad["roc_auc"]) < 1e-12
assert metrics_calibrated["brier"] < metrics_bad["brier"]
assert metrics_calibrated["log_loss"] < metrics_bad["log_loss"]
assert sum(report_selected[k] for k in ("tn", "fp", "fn", "tp")) == len(y_test)

print("Todas as verificações passaram.")
print(
    f"Ranking: ROC-AUC={metrics_base['roc_auc']:.6f}; "
    f"AP={metrics_base['average_precision']:.6f}; prevalência={y_test.mean():.6f}."
)
print(
    f"Distorção: Brier {metrics_base['brier']:.6f} → {metrics_bad['brier']:.6f}; "
    f"recalibração → {metrics_calibrated['brier']:.6f}."
)
print(
    f"Política: threshold={selected_threshold:.3f}; custo no teste "
    f"{report_default['mean_cost']:.6f} → {report_selected['mean_cost']:.6f}."
)

## Interpretação

- A transformação monotônica manteve o ranking, confirmando que ROC-AUC e AP não testam calibração.
- Brier e log-loss detectaram a distorção dos valores probabilísticos.
- O calibrador, ajustado apenas na validação, melhorou os proper scores no teste sem alterar substancialmente o ranking.
- A política orientada a custo trocou falsos positivos por menos falsos negativos e superou o threshold 0,5 no custo declarado.
- Os números valem para este processo sintético. Mudanças de prevalência, custos ou distribuição exigem revalidação.

**Desafio:** repita o experimento com outra prevalência apenas no teste. Observe que ROC-AUC pode permanecer parecida enquanto AP, precision, calibração e carga operacional mudam.
